### Q:1 Import the required libraries and load the Netflix dataset CSV file into a Pandas DataFrame named df. Display the first 10 rows using the appropriate method. Also print the total number of rows and columns.

In [ ]:
import pandas as pd

df = pd.read_csv('netflix_titles.csv')
df.head(10)

In [ ]:
df.info()

### Q2: Use a single Pandas method to get a comprehensive statistical summary of the DataFrame. What does this method reveal about the release_year column? State the minimum, maximum, and mean release year from the output.

In [ ]:
df.describe()

### Q:3 Check the data types of all columns in the DataFrame. Identify which columns are stored as object (string) type versus numeric. Which column would ideally benefit from being converted to a datetime type? Write the code to perform this conversion.

In [ ]:
df.dtypes

### Q:4 Find the total number of missing (null) values in each column of the DataFrame. Display these counts in descending order. Which three columns have the highest number of missing values?

In [ ]:
df1 = df.isnull().sum()
df1.sort_values(ascending=False)

### Q:5 Display a concise summary of the DataFrame using the .info() method. How many non-null entries does the director column have? What does this tell you about the quality of this column?

In [ ]:
df.info()

### Q:6 Count the total number of duplicate rows in the DataFrame. Drop all duplicate rows (if any exist), reset the index, and confirm the new shape of the cleaned DataFrame.

In [ ]:
df.duplicated().sum()

### Q:7 Using the type column, count how many entries are Movies versus TV Shows. Express each count as both a raw number and a percentage of the total dataset. Which content type dominates Netflix?

In [ ]:
df['type'].value_counts()
# movies dominate tv shows

# Section B:
### Data Cleaning & Handling Missing Values

### Q:8 Fill missing values in the director column with the string "Unknown", missing values in the country column with "Not Specified", and drop rows where the date_added column is null. Verify the result by re-checking null counts.

In [ ]:
df['director'].fillna('Unknown', inplace=True)

In [ ]:
df['country'].fillna("Not Specified", inplace=True)

In [ ]:
df.dropna(inplace=True)

In [ ]:
df.isnull().sum()

### Q:9 Extract the numeric duration value from the duration column (e.g., extract 90 from "90 min" and 2 from "2 Seasons"). Create a new column called duration_int with these integer values.

In [ ]:
df['duration_int'] = df['duration'].str.split(' ').str[0].astype(int)

### Q:10 Extract the year and month from the date_added column (after converting it to datetime) and create two new columns: year_added and month_added. Print the first 5 rows to verify.

In [ ]:
df['year'] = df['date_added'].str.extract(r'(\d{4})').astype(int)

In [ ]:
df['month'] = df['date_added'].str.extract('([a-zA-Z]+)')

In [ ]:
df.head()

### Q:11 Identify all unique values in the rating column. Are there any unusual or non-standard rating entries? If yes, replace any non-standard entries (such as numeric values that appear as ratings) with NaN and then drop those rows.

In [ ]:
df['rating'].unique()

In [ ]:
df['rating']
# there is no unordinary values in rating

### Q:12 Using NumPy, replace all remaining NaN values in the duration_int column (if any) with the median of that column. Use np.nanmedian() for this computation and verify with a null count.

In [ ]:
import numpy as np

median_val = np.nanmedian(df['duration_int'])

df['duration_int'].fillna(median_val, inplace=True)

In [ ]:
df['duration_int']

### Q:13 Create a new Boolean column called is_recent that is True if the release_year is 2018 or later, and False otherwise. Count how many titles are "recent" and how many are "older".

In [ ]:
df['is_recent'] = df['release_year'] >= 2018

In [ ]:
df['is_recent'].value_counts()

### Q:14 Rename the columns listed_in to genre and show_id to id in the DataFrame. Confirm the rename by printing the updated list of column names.

In [ ]:
df.rename(columns={'listed_in': 'genre', 'show_id': 'id'}, inplace=True)

# Section C:
### NumPy Operations & Statistical Analysis

### Q:15 Convert the release_year column to a NumPy array. Using only NumPy functions (no Pandas), compute the mean, median, standard deviation, and variance of this array. Print all four values.

In [ ]:
arr = df['release_year'].to_numpy()

In [ ]:
arr_mean = np.mean(arr)

In [ ]:
df['release_year'].fillna(arr_mean, inplace=True)

### Q:16 Using NumPy, compute the 25th percentile, 50th percentile, and 75th percentile of the duration_int column (for Movies only). Interpret what these values mean in context.

In [ ]:
movies = df[df['type']=='Movie']
per = np.percentile(movies['duration_int'], [25, 50, 75])

labels = ["25th percentile", "50th percentile (median)", "75th percentile"]

for l, p in zip(labels, per):
    print(f"{l}: {p}")

### Q:17 Create a NumPy array of the release_year values. Using NumPy boolean indexing, find all titles released between 2015 and 2020 (inclusive). Count how many such titles exist.

In [ ]:
arr = df['release_year'].to_numpy()
mask = (arr >= 2015) & (arr <= 2020)
arr[mask]
mask.sum()


### Q:18 Using NumPy, detect outliers in the duration_int column (Movies only) using the IQR method. Any value below Q1 − 1.5×IQR or above Q3 + 1.5×IQR is an outlier. How many outliers exist?

In [ ]:
import numpy as np

arr = (df[df['type'] == 'Movie']['duration_int']).to_numpy()

Q1 = np.percentile(arr, 25) 
Q3 = np.percentile(arr, 75)
IQR = Q3 - Q1

outlier_mask = (arr < Q1 - 1.5*IQR) | (arr > Q3 + 1.5*IQR)

outliers = arr[outlier_mask]

print("Number of outliers:", len(outliers))
print("Outlier values:", outliers)


### Q:19 Create a 2D NumPy array (matrix) where each row represents a year from 2015 to 2021, and the two columns represent the count of Movies and TV Shows added that year. Display this matrix and label the rows and columns clearly.

In [ ]:
import pandas as pd

group = df.groupby(['year', 'type']).size().unstack(fill_value=0)

result = group.loc[(group.index >= 2015) & (group.index <= 2020)]

result


### Q:20 Using NumPy, normalise the release_year column using Min-Max Normalisation so that all values fall between 0 and 1. Store the result in a new column called year_normalized and display the first 5 values.

In [ ]:
arr = df['release_year'].to_numpy()
normal = (arr - arr.min()) / (arr.max() - arr.min())
df['year_normalized'] = normal.round(2)
df

### Q:21 Compute the correlation between release_year and duration_int for Movies using np.corrcoef(). Interpret the resulting correlation coefficient: is there a positive, negative, or negligible relationship?

In [ ]:
movies = df[df["type"] == "Movie"]

years = movies["release_year"].to_numpy()
durations = movies["duration_int"].to_numpy()
np.corrcoef(years, durations)

corr_matrix = np.corrcoef(years, durations)

corr = corr_matrix[0, 1]

print("Correlation coefficient:", corr)

### Q:22 Find the top 10 countries that have produced the most content on Netflix. Display the result as a Pandas Series with country name and count. Which single country leads?

In [ ]:
countries = df['country'].dropna().str.split(',').explode().str.strip()

top_countries = countries.value_counts().head(10)

top_countries


### Q:23 Using groupby(), calculate the average movie duration (in minutes) for each content rating category (e.g., TV-MA, TV-14, PG-13). Sort the result in descending order of average duration.

In [ ]:
movies = df[df['type'] == 'Movie']

avg_duration = movies.groupby('rating')['duration_int'].mean()

avg_duration_sorted = avg_duration.sort_values(ascending=False)

avg_duration_sorted


### Q:24 Which year saw the highest number of titles added to Netflix? Use groupby() or value_counts() on the year_added column. Plot or describe the trend from 2015 to 2021.

In [ ]:
year_counts = df['year'].value_counts().sort_index()

peak_year = year_counts.idxmax()
peak_count = year_counts.max()

print(f"Year with most titles: {peak_year} ( {peak_count} titles )")

trend = year_counts.loc[2015:2021]
trend


### Q:25 Using the agg() method, compute the following statistics for duration_int grouped by type (Movie / TV Show): count, mean, min, max, and std. Display as a single aggregated table.

In [ ]:
stats = df.groupby('type')['duration_int'].agg(
    count='count',
    mean='mean',
    min='min',
    max='max',
    std='std'
)

stats


### Q:26 Filter the DataFrame to return only United States content that has a rating of TV-MA and was released after 2015. How many such titles exist? Display their titles and release years.

In [ ]:
us_tvma = df[
    (df["country"].str.contains("United States", na=False)) &
    (df["rating"] == "TV-MA") &
    (df["release_year"] > 2015)
]

count_titles = us_tvma.shape[0]
print("Number of titles:", count_titles)

us_tvma[["title", "release_year"]]


### Q:27 Using pivot_table(), create a pivot that shows the average release year for each combination of type (rows) and rating (columns). Fill missing combinations with 0. Display the result.

In [ ]:
pivot = df.pivot_table(
    values='release_year',
    index='type',
    columns='rating',
    aggfunc='mean',
    fill_value=0
)

pivot


### Q:28 Apply a custom function using .apply() that categorises each movie's duration into three groups: Short (less than 90 min), Medium (90–120 min), and Long (more than 120 min). Create a new column duration_category with these labels.

In [ ]:
def categorize_duration(x):
    if x < 90:
        return "Short"
    elif 90 <= x <= 120:
        return "Medium"
    else:
        return "Long"
    
df.loc[df['type'] == 'Movie', 'duration_category'] = df.loc[df['type'] == 'Movie', 'duration_int'].apply(categorize_duration)

df[['title', 'type', 'duration_int', 'duration_category']].head(10)


# Section E:
### Advanced EDA & Visualisation Interpretation

### Q:29 Using matplotlib or seaborn, plot a bar chart showing the count of Movies vs TV Shows. Add a title, axis labels, and different colors for each bar. Save the chart as type_distribution.png.

In [ ]:
import matplotlib.pyplot as plt

type_counts = df['type'].value_counts()

plt.figure(figsize=(6,4))
plt.bar(type_counts.index, type_counts.values, color=['skyblue', 'salmon'])

plt.title("Distribution of Content Types on Netflix")
plt.xlabel("Type")
plt.ylabel("Count")

plt.savefig("type_distribution.png", dpi=300, bbox_inches='tight')

plt.show()


### Q:30 Plot a histogram of the release_year column with 20 bins. Describe the shape of the distribution. Is it left-skewed, right-skewed, or approximately normal? Justify using the mean and median values.

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8,5))
plt.hist(df['release_year'], bins=20, color='skyblue', edgecolor='black')

plt.title("Distribution of Release Years on Netflix")
plt.xlabel("Release Year")
plt.ylabel("Count")

plt.show()

mean_year = df['release_year'].mean()
median_year = df['release_year'].median()

print("Mean release year:", mean_year)
print("Median release year:", median_year)


### Q:31 Create a box plot comparing the duration_int distribution of Movies versus TV Shows side by side using seaborn. Identify and label which group has more outliers based on the plot.

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(8,6))

sns.boxplot(x='type', y='duration_int', data=df, palette=['skyblue','salmon'])

plt.title("Distribution of Duration: Movies vs TV Shows")
plt.xlabel("Content Type")
plt.ylabel("Duration (minutes or seasons)")

plt.show()


### Q:32 Plot a line chart showing the total number of titles added to Netflix per year from 2015 to 2021. Add data point markers on each year. What trend do you observe, and what might explain the dip in 2020–2021?

In [ ]:
import matplotlib.pyplot as plt

year_counts = df['year'].value_counts().sort_index()

trend = year_counts.loc[2015:2021]

plt.figure(figsize=(8,5))
plt.plot(trend.index, trend.values, marker='o', linestyle='-', color='skyblue')

plt.title("Total Titles Added to Netflix (2015–2021)")
plt.xlabel("Year")
plt.ylabel("Number of Titles")
plt.grid(True)

plt.savefig("titles_per_year.png", dpi=300, bbox_inches='tight')

plt.show()

print(trend)


### Q:33 Using the genre (formerly listed_in) column, find the top 10 most common genres. Note that each entry can have multiple genres separated by commas — you must split and count each genre individually. Display as a horizontal bar chart.

In [ ]:
df.rename(columns={'listed_in': 'genre', 'show_id': 'id'}, inplace=True)

In [ ]:
import matplotlib.pyplot as plt

genres = df['genre'].dropna().str.split(',').explode().str.strip()

top_genres = genres.value_counts().head(10)

plt.figure(figsize=(8,6))
top_genres.plot(kind='barh', color='skyblue', edgecolor='black')

plt.title("Top 10 Most Common Netflix Genres")
plt.xlabel("Count")
plt.ylabel("Genre")

plt.gca().invert_yaxis()
plt.tight_layout()

plt.show()


### Q:34 Create a heatmap using seaborn showing the number of titles added each month (x-axis) for each year (y-axis) from 2015–2021. Which month consistently has the highest number of additions? Discuss the seasonal pattern.

In [ ]:
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

monthly_counts = df.groupby(['year','month']).size().reset_index(name='count')

pivot = monthly_counts.pivot(index='year', columns='month', values='count').fillna(0)

plt.figure(figsize=(10,6))
sns.heatmap(pivot, cmap="Blues", annot=True, fmt=".0f")

plt.title("Number of Titles Added to Netflix (2015–2021)")
plt.xlabel("Month")
plt.ylabel("Year")

plt.show()


### Q:35 Capstone Synthesis Question. Write a concise EDA summary report (minimum 10 lines of markdown comments in your notebook) covering: (a) dataset overview, (b) key data quality issues found, (c) three most interesting findings from your analysis, and (d) two business recommendations for Netflix based on your EDA insights.

# EDA Summary Report

1. **Dataset Overview**  
   - The dataset contains Netflix titles with attributes such as type (Movie/TV Show), title, director, cast, country, releas year, rating, duration, and genres (listed_in).  
   - It spans multiple years, with most entries concentrated between 2010–2021.  

2. **Key Data Quality Issues**  
   - Missing values in critical columns like `director`, `cast`, `country`, and `date_added`.  
   - Inconsistent formatting in `country` and `listed_in` (multiple values separated by commas).  
   - Duration column mixes minutes (Movies) and seasons (TV Shows), requiring transformation.  

3. **Interesting Findings**  
   - The **United States** leads by a wide margin in total content production.  
   - **2019** was the peak year for new titles added, followed by a decline in 2020–2021 (pandemic impact).  
   - **Drama** and **Comedy** dominate as the most comon genres, while **TV-MA** is the most frequent rating, reflecting Netflix focus on mature audiences.  

4. **Business Recommendations**  
   - **Diversify content sources**: Increase investment in non-U.S. markets (e.g., India, South Korea) where demand for local originals is rising.  
   - **Balance genre strategy**: While Drama/Comedy dominate, Netflix could expand niche genres (e.g., documentaries, family content) to attract underserved segments

5. **Overall Insight**  
   - Netflix catalog growth was rapid until 2019 then slowed due to external factors.  
   - The platforms strength lies in mature-rated dramas and comedies but opportunities exist in geographic and genre diversification.  
